### Setup

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import os

from models import Autoencoder, K_Sparse_Autoencoder, FC_WTA_Autoencoder
from datasets import get_data_loader, get_flattened_size

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset    = "mnist"
epochs     = 5
batch_size = 128
lr         = 1e-3
k          = 0.05

### Dataset

In [ ]:
os.makedirs('../data', exist_ok=True)

train_loader = get_data_loader(dataset, train=True,  batch_size=batch_size)
test_loader  = get_data_loader(dataset, train=False, batch_size=batch_size)

### Architecture

In [ ]:
def get_autoencoder_bottleneck_sizes(dataset: str) -> list:
    if dataset == "mnist":
        return [256, 64, 16, 4]
    elif dataset == "cifar10":
        return [256, 64, 16, 4]
    else:
        raise ValueError(f"No bottleneck schedule defined for {dataset!r}")

### Chain Training

In [ ]:
input_dim        = get_flattened_size(dataset)
bottleneck_sizes = get_autoencoder_bottleneck_sizes(dataset)
all_dims         = [input_dim] + bottleneck_sizes  # e.g. [784, 256, 64, 16, 4]

def make_autoencoder(model_type, in_dim, out_dim):
    if model_type == "normal":
        return Autoencoder(dim=(in_dim, out_dim))
    elif model_type == "k_sparse":
        return K_Sparse_Autoencoder(dim=(in_dim, out_dim), k=k, total_epochs=epochs, a=1)
    elif model_type == "fc_wta":
        return FC_WTA_Autoencoder(dim=(in_dim, out_dim), k=k)
    else:
        raise ValueError(f"Unknown model type: {model_type!r}")

model_types = ["normal", "k_sparse", "fc_wta"]

chains       = {}
chain_losses = {}

model_bar = tqdm(model_types, desc="Model")
for model_type in model_bar:
    model_bar.set_description(f"Model: {model_type}")

    autoencoders     = []
    all_layer_losses = []
    current_loader   = train_loader

    layer_pairs = list(zip(all_dims, all_dims[1:]))
    layer_bar   = tqdm(enumerate(layer_pairs), total=len(layer_pairs), desc="Layer", leave=False)
    for layer_idx, (in_dim, out_dim) in layer_bar:
        layer_bar.set_description(f"Layer {layer_idx + 1}: {in_dim}→{out_dim}")

        autoencoder = make_autoencoder(model_type, in_dim, out_dim).to(device)
        optimizer   = torch.optim.Adam(autoencoder.parameters(), lr=lr)
        criterion   = nn.MSELoss()
        layer_losses = []
        autoencoder.train()

        # Train current layer
        epoch_bar = tqdm(range(1, epochs + 1), desc="Epoch", leave=False)
        for epoch in epoch_bar:
            epoch_loss = 0.0
            for batch_inputs, _ in current_loader:
                batch_inputs = batch_inputs.to(device)
                optimizer.zero_grad()
                x_recon = autoencoder(batch_inputs, epoch=epoch)
                loss    = criterion(x_recon, batch_inputs)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            avg = epoch_loss / len(current_loader)
            layer_losses.append(avg)
            epoch_bar.set_postfix(loss=f"{avg:.4f}")

        all_layer_losses.append(layer_losses)
        autoencoders.append(autoencoder)

        # Build representations for the next layer
        autoencoder.eval()
        all_reps, all_labels = [], []
        with torch.no_grad():
            for batch_inputs, batch_labels in current_loader:
                z = autoencoder.encode(batch_inputs.to(device))
                all_reps.append(z.cpu())
                all_labels.append(batch_labels)

        reps           = torch.cat(all_reps,   dim=0)
        labels         = torch.cat(all_labels, dim=0)
        current_loader = DataLoader(
            TensorDataset(reps, labels),
            batch_size=batch_size, shuffle=True, pin_memory=True
        )

    chains[model_type]       = autoencoders
    chain_losses[model_type] = all_layer_losses

### Visualization

In [ ]:
n_layers = len(all_dims) - 1
fig, axes = plt.subplots(len(model_types), n_layers, figsize=(5 * n_layers, 4 * len(model_types)))

for row, model_type in enumerate(model_types):
    for col in range(n_layers):
        ax      = axes[row][col]
        losses  = chain_losses[model_type][col]
        in_dim  = all_dims[col]
        out_dim = all_dims[col + 1]
        ax.plot(range(1, epochs + 1), losses, "o-")
        ax.set_title(f"{model_type} | Layer {col + 1} ({in_dim}→{out_dim})")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Build Representation Dataset

In [ ]:
ordered_loader = get_data_loader(dataset, train=True, batch_size=batch_size)

raw_inputs, raw_labels = [], []
for batch_inputs, batch_labels in ordered_loader:
    raw_inputs.append(batch_inputs)
    raw_labels.append(batch_labels)

raw_tensor    = torch.cat(raw_inputs, dim=0)
labels_tensor = torch.cat(raw_labels, dim=0)

rep_datasets = {}

for model_type in model_types:
    all_layer_reps = [raw_tensor]
    current_reps   = raw_tensor

    for ae in chains[model_type]:
        ae.eval()
        next_reps  = []
        rep_loader = DataLoader(TensorDataset(current_reps), batch_size=batch_size)
        with torch.no_grad():
            for (batch,) in rep_loader:
                z = ae.encode(batch.to(device))
                next_reps.append(z.cpu())
        current_reps = torch.cat(next_reps, dim=0)
        all_layer_reps.append(current_reps)

    rep_datasets[model_type] = TensorDataset(*all_layer_reps, labels_tensor)

    shapes = ", ".join(
        f"{'raw' if i == 0 else f'L{i}'}:{tuple(r.shape)}"
        for i, r in enumerate(all_layer_reps)
    )
    tqdm.write(f"[{model_type}] {shapes}")

In [ ]:
save_dir = '../results/chained_sparse_autoencoders'
os.makedirs(save_dir, exist_ok=True)

for model_type in model_types:
    ds = rep_datasets[model_type]
    torch.save(
        {'layer_reps': list(ds.tensors[:-1]), 'labels': ds.tensors[-1]},
        f'{save_dir}/{dataset}_{model_type}_representations.pt'
    )
    torch.save(
        [ae.state_dict() for ae in chains[model_type]],
        f'{save_dir}/{dataset}_{model_type}_autoencoders.pt'
    )

print(f"[saved] {save_dir}/")